# EEG Preprocessing Debug Pipeline

Step-by-step walkthrough of `src/eeg_preprocess.py` with visualizations at each stage.  
Uses the **exact same code** as the production pipeline — no simplifications.

**Usage:** Set `SJ_NUM`, `CONDITION_IDX`, and `DATA_DIR` in the next cell, then Run All.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    USER CONFIG                              ║
# ╚══════════════════════════════════════════════════════════════╝

SJ_NUM = 4                # Subject number
CONDITION_IDX = 0         # 0=sit_attend, 1=sit_unattend, 2=walk_attend, 3=walk_unattend
DATA_DIR = "/Users/madhav/PSY197B/data"  # Path to data/ folder

# ── Pipeline parameters (from run_config.yaml) ──────────────────
SFREQ_TARGET = 250
FILTER_LOW = 0.5
FILTER_HIGH = 30.0
REF_CHANNELS = ["TP9", "TP10"]
BAD_CHAN_Z_THRESH = 3.5
DETECT_BAD_CHANNELS = True
APPLY_ICA = False
USE_GEDAI = True
TRIGGER_LATENCY_OFFSET = 60
TMIN = -0.2
TMAX = 1.0
BASELINE = (-0.2, 0)
TARGET_CHANNELS = ["Pz"]
ERP_CODES = {1, 2, 3}

CONDITIONS = [
    {"eeg_label": "sit_attend",    "trial_label": "Attend_Sit"},
    {"eeg_label": "sit_unattend",  "trial_label": "Unattend_Sit"},
    {"eeg_label": "walk_attend",   "trial_label": "Attend_Walk"},
    {"eeg_label": "walk_unattend", "trial_label": "Unattend_Walk"},
]

# Hard-coded trial removals (MATLAB parity for sj01-04)
_MANUAL_TRIAL_DROPS_1BASED = {
    (1, "sit_attend"):     [753],
    (1, "sit_unattend"):   [742, 939],
    (1, "walk_attend"):    [341, 648, 801, 974, 995],
    (1, "walk_unattend"):  [130, 133, 295],
    (2, "walk_attend"):    [888],
    (2, "walk_unattend"):  [434, 601, 872],
    (4, "sit_unattend"):   [984],
}

# ── Derived ──────────────────────────────────────────────────────
cond = CONDITIONS[CONDITION_IDX]
label = cond["eeg_label"]
source_dir_eeg = f"{DATA_DIR}/sj{SJ_NUM:02d}/eeg"
source_dir_trial = f"{DATA_DIR}/sj{SJ_NUM:02d}/beh"

print(f"Subject: sj{SJ_NUM:02d}")
print(f"Condition: {label} ({cond['trial_label']})")
print(f"EEG dir: {source_dir_eeg}")
print(f"BEH dir: {source_dir_trial}")

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
mne.set_log_level("WARNING")
plt.rcParams['figure.dpi'] = 100

## 1. Load Behavioral Data

Loads 5 block CSVs per condition, concatenates into one DataFrame.

In [ ]:
# ── Load behavioural trial data (same as eeg_preprocess.py:88-103) ──
trial_data_list = []
for i_block in range(1, 6):
    filename = os.path.join(
        source_dir_trial,
        f"sj{SJ_NUM:02d}_block{i_block}_{cond['trial_label']}.csv",
    )
    if os.path.exists(filename):
        trial_data_list.append(pd.read_csv(filename))
    else:
        print(f"    Warning: No file for block {i_block}: {filename}")

if not trial_data_list:
    raise FileNotFoundError(f"No trial data for {cond['trial_label']}")

trial_data = pd.concat(trial_data_list, ignore_index=True)
print(f"Loaded {len(trial_data)} trials from {len(trial_data_list)} blocks")

In [ ]:
# ── DIAGNOSTIC: Inspect behavioral data ──
print(f"Shape: {trial_data.shape}")
print(f"Columns: {list(trial_data.columns)}")
print(f"\ntrialIdx range: {trial_data['trialIdx'].min()} – {trial_data['trialIdx'].max()}")
print(f"trialType value counts:\n{trial_data['trialType'].value_counts().to_string()}")
if 'outcome' in trial_data.columns:
    print(f"\nOutcome distribution:\n{trial_data['outcome'].value_counts().to_string()}")
display(trial_data.head(10))

## 2. Manual Trial Removals (MATLAB Parity, sj01–04)

In [ ]:
def remove_trials_matlab_style(df, indices_1based):
    """Remove rows using MATLAB's sequential 1-based deletion semantics.

    Drops in descending order so each removal doesn't shift the indices
    of subsequent removals — identical to MATLAB's repeated
    trialData(row,:)=[] pattern.
    """
    if not indices_1based:
        return df
    for idx in sorted([i - 1 for i in indices_1based], reverse=True):
        if idx < len(df):
            df = df.drop(df.index[idx]).reset_index(drop=True)
    return df


# ── Apply manual trial removals (same as eeg_preprocess.py:107-112) ──
n_before = len(trial_data)
drop_key = (SJ_NUM, label)
if drop_key in _MANUAL_TRIAL_DROPS_1BASED:
    idxs = _MANUAL_TRIAL_DROPS_1BASED[drop_key]
    trial_data = remove_trials_matlab_style(trial_data, idxs)
    print(f"Manual trial removal ({label}): dropped {len(idxs)} trials")
    print(f"  Indices (1-based): {idxs}")
    print(f"  Rows: {n_before} → {len(trial_data)}")
else:
    print(f"No manual drops for sj{SJ_NUM:02d}/{label}")

## 3. Load Raw EEG

In [ ]:
# ── Load raw BrainVision EEG (same as eeg_preprocess.py:115-121) ──
eeg_file = os.path.join(source_dir_eeg, f"sj{SJ_NUM:02d}_{label}.vhdr")
if not os.path.exists(eeg_file):
    raise FileNotFoundError(f"EEG file not found: {eeg_file}")

print(f"Loading: {eeg_file}")
raw = mne.io.read_raw_brainvision(eeg_file, preload=True)

In [ ]:
# ── DIAGNOSTIC: Raw EEG info ──
print(raw.info)
print(f"\nChannels ({raw.info['nchan']}): {raw.ch_names}")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.n_times / raw.info['sfreq']:.1f} s ({raw.n_times} samples)")
print(f"\nAnnotations ({len(raw.annotations)}):")
if len(raw.annotations) > 0:
    unique_desc = pd.Series([a for a in raw.annotations.description]).value_counts()
    print(unique_desc.to_string())

# Plot 5s of raw data
raw.plot(duration=5, n_channels=20, scalings='auto', title=f"Raw EEG — sj{SJ_NUM:02d} {label}")
plt.show()

## 4. Montage Correction (sj01–04 only)

In [ ]:
def load_correct_montage_for_early_subjects(raw, sj_num):
    """Remap channel names/positions for sj01-04 using a reference cap file."""
    if sj_num > 4:
        return raw
    ref_path = os.path.join(
        DATA_DIR, "Dependencies",
        "EEG_32ch_Cap_Correct_Montage", "Test_32ch.vhdr",
    )
    if not os.path.exists(ref_path):
        print(f"    Warning: reference montage not found ({ref_path}) — skipping")
        return raw
    ref = mne.io.read_raw_brainvision(ref_path, preload=False, verbose=False)
    rename_map = dict(zip(raw.ch_names, ref.ch_names))
    raw.rename_channels(rename_map)
    if ref.get_montage() is not None:
        raw.set_montage(ref.get_montage(), on_missing="warn")
    print(f"    Montage corrected for sj{sj_num:02d} using {ref_path}")
    return raw


# ── Apply montage correction (same as eeg_preprocess.py:124) ──
ch_names_before = raw.ch_names.copy()
raw = load_correct_montage_for_early_subjects(raw, SJ_NUM)

# DIAGNOSTIC
if SJ_NUM <= 4:
    changed = [f"{a} → {b}" for a, b in zip(ch_names_before, raw.ch_names) if a != b]
    if changed:
        print(f"Channel renames ({len(changed)}):")
        for c in changed[:10]:
            print(f"  {c}")
    montage = raw.get_montage()
    if montage:
        fig = montage.plot(show=False)
        plt.title(f"Corrected montage — sj{SJ_NUM:02d}")
        plt.show()
else:
    print(f"sj{SJ_NUM:02d} > 04: no montage correction needed")

## 5. Downsample

In [ ]:
# ── Downsample (same as eeg_preprocess.py:126-128) ──
if raw.info["sfreq"] != SFREQ_TARGET:
    print(f"Downsampling from {raw.info['sfreq']:.1f} Hz to {SFREQ_TARGET} Hz")
    raw = raw.resample(SFREQ_TARGET)
else:
    print(f"Already at {SFREQ_TARGET} Hz — no resampling needed")

print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"N samples: {raw.n_times}")

## 6. Re-reference (TP9/TP10 average)

In [ ]:
# ── Re-reference (same as eeg_preprocess.py:130-134) ──
if all(ch in raw.ch_names for ch in REF_CHANNELS):
    print(f"Re-referencing to average of {REF_CHANNELS}")
    raw.set_eeg_reference(REF_CHANNELS, ch_type="eeg")
else:
    missing = [ch for ch in REF_CHANNELS if ch not in raw.ch_names]
    print(f"Warning: reference channels not found ({missing}), skipping re-reference")

## 7. Remove Motion Tracking Channels

In [ ]:
# ── Remove motion tracking channels (same as eeg_preprocess.py:136-148) ──
ch_to_remove = [
    ch
    for ch in raw.ch_names
    if any(
        x in ch.lower()
        for x in ["x_dir", "y_dir", "z_dir", "r_x", "r_y", "r_z",
                   "l_x", "l_y", "l_z"]
    )
    or ch == "32"
]
if ch_to_remove:
    print(f"Removing {len(ch_to_remove)} channels: {ch_to_remove}")
    raw.drop_channels(ch_to_remove)
else:
    print("No motion tracking channels found")

print(f"\nRemaining channels ({raw.info['nchan']}): {raw.ch_names}")

## 8. Bandpass Filter (0.5–30 Hz)

In [ ]:
# ── Bandpass filter (same as eeg_preprocess.py:150-151) ──
# Store pre-filter segment for comparison
ch_plot = "Cz" if "Cz" in raw.ch_names else raw.ch_names[0]
start_samp = int(10 * raw.info['sfreq'])
n_samp = int(3 * raw.info['sfreq'])
pre_filt_segment = raw.get_data(picks=[ch_plot])[0, start_samp:start_samp+n_samp].copy()

print(f"Filtering: {FILTER_LOW}–{FILTER_HIGH} Hz (FIR, firwin2)")
raw.filter(FILTER_LOW, FILTER_HIGH, fir_design="firwin2")

In [ ]:
# ── DIAGNOSTIC: Before/after filter comparison ──
post_filt_segment = raw.get_data(picks=[ch_plot])[0, start_samp:start_samp+n_samp]
t = np.arange(n_samp) / raw.info['sfreq']

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(t, pre_filt_segment * 1e6, linewidth=0.5)
axes[0].set_title(f"{ch_plot} BEFORE filter")
axes[0].set_ylabel("uV")

axes[1].plot(t, post_filt_segment * 1e6, linewidth=0.5)
axes[1].set_title(f"{ch_plot} AFTER {FILTER_LOW}–{FILTER_HIGH} Hz filter")
axes[1].set_ylabel("uV")
axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

# PSD
fig, ax = plt.subplots(figsize=(10, 4))
raw.compute_psd(fmax=50).plot(axes=ax, show=False)
ax.set_title("Power Spectral Density (post-filter)")
plt.tight_layout()
plt.show()

del pre_filt_segment, post_filt_segment

## 9. Bad Channel Detection

In [ ]:
# ── Bad-channel detection (same as eeg_preprocess.py:154-168) ──
bad_chans = []
z_scores = np.array([])
chan_std = np.array([])
eeg_picks = np.array([])

if DETECT_BAD_CHANNELS:
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    if len(eeg_picks) > 0:
        data_eeg = raw.get_data(picks=eeg_picks)
        chan_std = np.std(data_eeg, axis=1)
        z_scores = (chan_std - np.mean(chan_std)) / (np.std(chan_std) + 1e-12)
        bad_chans = [
            raw.ch_names[eeg_picks[i]]
            for i, z in enumerate(z_scores)
            if z > BAD_CHAN_Z_THRESH
        ]
        if bad_chans:
            print(f"Marking bad channels (z>{BAD_CHAN_Z_THRESH}): {bad_chans}")
            raw.info["bads"].extend(bad_chans)
            raw.interpolate_bads(reset_bads=True)
        else:
            print("No bad channels detected")
else:
    print("Bad-channel detection: disabled (detect_bad_channels=false)")

In [ ]:
# ── DIAGNOSTIC: Bad channel detection visualization ──
if DETECT_BAD_CHANNELS and len(z_scores) > 0:
    eeg_ch_names = [raw.ch_names[i] for i in eeg_picks]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Z-score histogram
    axes[0].hist(z_scores, bins=20, edgecolor='k', alpha=0.7)
    axes[0].axvline(BAD_CHAN_Z_THRESH, color='r', linestyle='--',
                    label=f'threshold = {BAD_CHAN_Z_THRESH}')
    axes[0].set_xlabel("Z-score of channel std")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Bad channel detection: z-score distribution")
    axes[0].legend()

    # Per-channel std bar chart
    colors = ['red' if z > BAD_CHAN_Z_THRESH else 'steelblue' for z in z_scores]
    axes[1].bar(range(len(chan_std)), chan_std * 1e6, color=colors)
    axes[1].set_xticks(range(len(eeg_ch_names)))
    axes[1].set_xticklabels(eeg_ch_names, rotation=90, fontsize=7)
    axes[1].set_ylabel("Std (uV)")
    axes[1].set_title("Per-channel standard deviation")

    plt.tight_layout()
    plt.show()

    print(f"\nBad channels: {bad_chans if bad_chans else 'None'}")
    print(f"Z-scores per channel:")
    for name, z in sorted(zip(eeg_ch_names, z_scores), key=lambda x: -x[1])[:10]:
        marker = " *** BAD" if z > BAD_CHAN_Z_THRESH else ""
        print(f"  {name:6s}: z={z:.2f}{marker}")
else:
    print("Bad channel detection was disabled or no EEG channels found")

## 10. Artifact Removal: GEDAI

Runs if `USE_GEDAI = True`. Leadfield-aware multiresolution artifact removal.

In [ ]:
# ── Artifact removal: GEDAI path (same as eeg_preprocess.py:173-181 + gedai_preprocess.py) ──
gedai = None

if USE_GEDAI:
    try:
        from gedai import Gedai
        GEDAI_AVAILABLE = True
    except ImportError:
        GEDAI_AVAILABLE = False
        print("WARNING: gedai package not installed — skipping GEDAI")

    if GEDAI_AVAILABLE:
        # Store pre-GEDAI data for comparison
        raw_before_gedai = raw.copy()

        # --- Inline GEDAI application (from gedai_preprocess.py:55-91) ---
        picks_eeg = mne.pick_types(raw.info, eeg=True, exclude="bads")
        n_ch = len(picks_eeg)

        if n_ch < 3:
            print(f"GEDAI: too few EEG channels ({n_ch}), skipping")
        else:
            if raw.get_montage() is None:
                print("GEDAI: no montage set — applying standard_1020")
                raw.set_montage("standard_1020", on_missing="warn")

            print(f"GEDAI: fitting on {n_ch} EEG channels (wavelet=haar, level=0, ref_cov=leadfield)")

            gedai = Gedai(
                wavelet_type="haar",
                wavelet_level=0,
            )

            gedai.fit_raw(
                raw,
                reference_cov="leadfield",
                sensai_method="optimize",
            )

            raw_clean = gedai.transform_raw(raw)

            # Replace EEG channels in-place
            picks_eeg = mne.pick_types(raw.info, eeg=True, exclude="bads")
            raw._data[picks_eeg] = raw_clean.get_data(picks=picks_eeg)

            print("GEDAI: transform complete")
else:
    print("GEDAI: disabled (use_gedai=false)")

In [ ]:
# ── DIAGNOSTIC: GEDAI before/after ──
if USE_GEDAI and 'raw_before_gedai' in dir():
    ch_plot = "Cz" if "Cz" in raw.ch_names else raw.ch_names[0]
    start_samp = int(10 * raw.info['sfreq'])
    n_samp = int(3 * raw.info['sfreq'])
    t = np.arange(n_samp) / raw.info['sfreq']

    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    axes[0].plot(t, raw_before_gedai.get_data(picks=[ch_plot])[0, start_samp:start_samp+n_samp] * 1e6, linewidth=0.5)
    axes[0].set_title(f"{ch_plot} BEFORE GEDAI")
    axes[0].set_ylabel("uV")

    axes[1].plot(t, raw.get_data(picks=[ch_plot])[0, start_samp:start_samp+n_samp] * 1e6, linewidth=0.5)
    axes[1].set_title(f"{ch_plot} AFTER GEDAI")
    axes[1].set_ylabel("uV")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

    # Difference signal
    fig, ax = plt.subplots(figsize=(14, 3))
    diff = (raw_before_gedai.get_data(picks=[ch_plot])[0, start_samp:start_samp+n_samp] -
            raw.get_data(picks=[ch_plot])[0, start_samp:start_samp+n_samp]) * 1e6
    ax.plot(t, diff, linewidth=0.5, color='red')
    ax.set_title(f"{ch_plot} REMOVED by GEDAI (before - after)")
    ax.set_ylabel("uV")
    ax.set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

    # GEDAI diagnostic plots
    if gedai is not None:
        try:
            result = gedai.plot_fit()
            if result is not None:
                figs = result if isinstance(result, (list, tuple)) else [result]
                for f in figs:
                    display(f)
                    plt.close(f)
        except Exception as e:
            print(f"GEDAI plot_fit failed: {e}")

    del raw_before_gedai

## 11. Artifact Removal: ICA (alternative path)

Runs if `APPLY_ICA = True` and `USE_GEDAI = False`.

In [ ]:
# ── Artifact removal: ICA path (same as eeg_preprocess.py:182-248) ──
ica = None
eog_indices = []
eog_channels = []

if not USE_GEDAI and APPLY_ICA:
    # Store pre-ICA for comparison
    raw_before_ica = raw.copy()

    print("ICA: fitting fastica (n_components=0.99 variance on EEG picks)...")
    ica = mne.preprocessing.ICA(
        n_components=0.99, method="fastica",
        random_state=97, max_iter="auto",
    )
    ica.fit(raw, picks="eeg")
    ica_component_indices = list(range(ica.n_components_))
    print(f"ICA: fit done — {ica.n_components_} components")

    eog_channels = [ch for ch in ["Fp1", "Fp2", "AF3", "AF4"]
                    if ch in raw.ch_names]
    eog_indices = []
    if eog_channels:
        for eog_ch in eog_channels:
            try:
                inds, _ = ica.find_bads_eog(raw, ch_name=eog_ch, verbose=False)
                eog_indices.extend(inds)
            except Exception:
                pass
        eog_indices = list(set(eog_indices))
        if eog_indices:
            print(f"ICA: EOG-based candidates — {len(eog_indices)} component(s): {sorted(eog_indices)}")
            ica.exclude = eog_indices
        else:
            print("ICA: find_bads_eog found no components to exclude")
    else:
        print("ICA: no Fp1/Fp2/AF3/AF4 — skipping automatic EOG detection")

    excluded = sorted(set(ica.exclude))
    print(f"ICA: ica.exclude = {excluded}")

    if excluded:
        print(f"ICA: projecting out {len(excluded)} component(s)")
    raw = ica.apply(raw)
    print("ICA: apply(raw) finished")
elif not USE_GEDAI and not APPLY_ICA:
    print("Artifact removal: disabled (both use_gedai and apply_ica are false)")
else:
    print("ICA: skipped (using GEDAI instead)")

In [ ]:
# ── DIAGNOSTIC: ICA visualization ──
if ica is not None:
    # Component topomaps
    print("ICA Component Topomaps:")
    n_maps = min(ica.n_components_, 20)
    figs = ica.plot_components(inst=raw, picks=list(range(n_maps)), show=False)
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for f in figs:
        display(f)
        plt.close(f)

    # EOG correlation
    if eog_channels:
        print(f"\nEOG correlation channels: {eog_channels}")
        print(f"Excluded components: {sorted(ica.exclude)}")

    # Before/after overlay
    if 'raw_before_ica' in dir() and ica.exclude:
        print("\nICA overlay (blue=before, red=after):")
        fig = ica.plot_overlay(raw_before_ica, exclude=ica.exclude, show=False)
        display(fig)
        plt.close(fig)
        del raw_before_ica
else:
    print("ICA was not run")

## 12. Event Extraction & Latency Adjustment

In [ ]:
# ── Event extraction + latency adjustment (same as eeg_preprocess.py:253-261) ──
events, event_id = mne.events_from_annotations(raw)

print(f"Adjusting trigger latencies (+{TRIGGER_LATENCY_OFFSET} samples for triggers <= 200)")
events_adjusted = events.copy()
events_adjusted[events_adjusted[:, 2] <= 200, 0] += TRIGGER_LATENCY_OFFSET

valid_events = events_adjusted[events_adjusted[:, 2] <= 200]
valid_event_ids = {str(code): code for code in np.unique(valid_events[:, 2])}

In [ ]:
# ── DIAGNOSTIC: Events ──
print(f"Total annotations found: {len(events)}")
print(f"All unique event codes: {sorted(np.unique(events[:, 2]))}")
print(f"\nValid events (code <= 200): {len(valid_events)}")
print(f"Valid event IDs: {valid_event_ids}")
print(f"\nLatency offset: +{TRIGGER_LATENCY_OFFSET} samples = "
      f"+{TRIGGER_LATENCY_OFFSET / raw.info['sfreq'] * 1000:.1f} ms")

# Event code histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].hist(valid_events[:, 2], bins=len(valid_event_ids), edgecolor='k')
axes[0].set_xlabel("Event code")
axes[0].set_ylabel("Count")
axes[0].set_title("Valid event code distribution")

# ISI histogram
isi = np.diff(valid_events[:, 0]) / raw.info['sfreq']
axes[1].hist(isi, bins=50, edgecolor='k')
axes[1].set_xlabel("Inter-stimulus interval (s)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"ISI distribution (median={np.median(isi):.2f}s)")
plt.tight_layout()
plt.show()

print(f"\nEvent code value counts:")
codes, counts = np.unique(valid_events[:, 2], return_counts=True)
for c, n in zip(codes, counts):
    print(f"  Code {c:3d}: {n} events")

## 13. Epoching

In [ ]:
# ── Epoching (same as eeg_preprocess.py:263-274) ──
epochs = mne.Epochs(
    raw,
    valid_events,
    event_id=valid_event_ids,
    tmin=TMIN,
    tmax=TMAX,
    baseline=None,
    preload=True,
    verbose=False,
)
epochs.apply_baseline(baseline=BASELINE)
print(f"Created {len(epochs)} epochs")

In [ ]:
# ── DIAGNOSTIC: Epoch info ──
print(f"Epochs: {len(epochs)}")
print(f"Time range: {epochs.tmin:.3f} to {epochs.tmax:.3f} s")
print(f"Baseline: {BASELINE}")
print(f"Event IDs: {epochs.event_id}")

# Drop log
n_good = sum(1 for d in epochs.drop_log if len(d) == 0)
n_bad = len(epochs.drop_log) - n_good
print(f"\nDrop log: {n_good} good, {n_bad} dropped")
if n_bad > 0:
    reasons = [d for d in epochs.drop_log if len(d) > 0]
    print(f"  Drop reasons (first 5): {reasons[:5]}")

# Grand average butterfly
fig = epochs.average().plot(show=False, spatial_colors=True)
plt.suptitle(f"Grand average (all codes) — sj{SJ_NUM:02d} {label}", y=1.02)
plt.show()

# Single-trial image for one channel
ch_img = TARGET_CHANNELS[0] if TARGET_CHANNELS[0] in epochs.ch_names else epochs.ch_names[0]
fig = epochs.plot_image(picks=[ch_img], show=False)[0]
plt.suptitle(f"Single-trial image: {ch_img}")
plt.show()

## 14. Trial ↔ EEG Alignment

In [ ]:
def align_to_eeg_events(df, eeg_event_list, idx_col="trialIdx"):
    """Align behavioral/ET dataframe to EEG event codes.

    Handles:
    - Exact match (all codes in same order) -> return as-is
    - BEH > EEG (greedy match) -> find each EEG code in BEH
    - BEH < EEG -> ERROR
    """
    import numpy as np

    beh_codes = df[idx_col].values
    eeg_codes = np.array(eeg_event_list)

    if len(beh_codes) == len(eeg_codes) and np.all(beh_codes == eeg_codes):
        print("    SYNC SUCCESS (exact match, all blocks)")
        return df.reset_index(drop=True)

    if len(beh_codes) > len(eeg_codes):
        print(f"    BEH has {len(beh_codes)} trials, EEG has {len(eeg_codes)} — trimming BEH")
        beh_ptr = 0
        aligned_rows = []
        for eeg_code in eeg_codes:
            while beh_ptr < len(beh_codes) and beh_codes[beh_ptr] != eeg_code:
                beh_ptr += 1
            if beh_ptr >= len(beh_codes):
                print("    SYNC FAIL — ran out of BEH trials")
                return None
            aligned_rows.append(beh_ptr)
            beh_ptr += 1

        aligned = df.iloc[aligned_rows].reset_index(drop=True)
        if np.all(aligned[idx_col].values == eeg_codes):
            print(f"    SYNC SUCCESS ({len(aligned)} trials aligned)")
            return aligned
        else:
            print("    SYNC FAIL — final check mismatch")
            return None

    if len(beh_codes) < len(eeg_codes):
        print(f"    BEH has fewer trials ({len(beh_codes)}) than EEG ({len(eeg_codes)})")
        print("    SYNC FAIL — cannot align")
        return None

    print("    SYNC FAIL — unknown alignment issue")
    return None


# ── Trial-EEG alignment (same as eeg_preprocess.py:277-309) ──
eeg_event_list = epochs.events[:, 2]

print(f"EEG epochs: {len(eeg_event_list)}")
print(f"BEH trials: {len(trial_data)}")

missing_in_eeg = set(trial_data["trialIdx"]) - set(eeg_event_list)
if missing_in_eeg:
    print(f"Removing {len(missing_in_eeg)} trials with missing EEG triggers")
    trial_data = trial_data[
        trial_data["trialIdx"].isin(eeg_event_list)
    ].reset_index(drop=True)

trial_data = align_to_eeg_events(trial_data, eeg_event_list)
if trial_data is None:
    print("ABORT — EEG/trial sync failed")
else:
    print(f"\nFinal aligned trial count: {len(trial_data)}")

In [ ]:
# ── DIAGNOSTIC: Alignment verification ──
if trial_data is not None:
    # Sync verification (same as eeg_preprocess.py:292-306)
    print("SYNC VERIFICATION:")
    print(f"  EEG epochs: {len(eeg_event_list)}")
    print(f"  Trial data rows: {len(trial_data)}")

    if len(trial_data) == len(eeg_event_list):
        mismatches = int(np.sum(trial_data["trialIdx"].values != eeg_event_list))
        print(f"  Event code mismatches: {mismatches}")
        if mismatches > 0:
            mismatch_idx = np.where(trial_data["trialIdx"].values != eeg_event_list)[0]
            print(f"  First mismatches at positions: {mismatch_idx[:10]}")
            for i in mismatch_idx[:5]:
                print(f"    Row {i}: BEH={trial_data['trialIdx'].values[i]}, EEG={eeg_event_list[i]}")

    # Attach metadata
    if len(trial_data) == len(epochs):
        epochs.metadata = trial_data.copy()
        print("\n  Metadata attached to epochs")

    # Code distribution comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].hist(trial_data['trialIdx'], bins=30, edgecolor='k')
    axes[0].set_title(f"BEH trialIdx ({len(trial_data)} trials)")
    axes[0].set_xlabel("trialIdx")
    axes[1].hist(eeg_event_list, bins=30, edgecolor='k', color='orange')
    axes[1].set_title(f"EEG event codes ({len(eeg_event_list)} epochs)")
    axes[1].set_xlabel("Event code")
    plt.tight_layout()
    plt.show()
else:
    print("ALIGNMENT FAILED — cannot proceed with ERP analysis")

## 15. Final ERPs

Split by event code (Go vs NoGo) at target channel(s).

In [ ]:
# ── Final ERP visualization ──
ch = TARGET_CHANNELS[0] if TARGET_CHANNELS[0] in epochs.ch_names else epochs.ch_names[0]
times = epochs.times

fig, ax = plt.subplots(figsize=(10, 5))
for code_str, code_val in sorted(epochs.event_id.items(), key=lambda x: x[1]):
    try:
        evoked = epochs[code_str].average()
        ch_idx = evoked.ch_names.index(ch)
        n_trials = len(epochs[code_str])
        ax.plot(times * 1000, evoked.data[ch_idx] * 1e6,
                label=f"Code {code_str} (n={n_trials})", linewidth=1.5)
    except Exception:
        pass

ax.axvline(0, color='k', linestyle='--', alpha=0.5, label='Stimulus onset')
ax.axhline(0, color='k', linestyle='-', alpha=0.3)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Amplitude (uV)")
ax.set_title(f"ERPs at {ch} — sj{SJ_NUM:02d} {label}")
ax.legend(fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# If metadata has trialType, also plot Go vs NoGo
if epochs.metadata is not None and 'trialType' in epochs.metadata.columns:
    fig, ax = plt.subplots(figsize=(10, 5))

    go_mask = epochs.metadata['trialType'] == 10
    nogo_mask = epochs.metadata['trialType'] == 20

    if go_mask.sum() > 0:
        go_evoked = epochs[go_mask].average()
        ch_idx = go_evoked.ch_names.index(ch)
        ax.plot(times * 1000, go_evoked.data[ch_idx] * 1e6,
                label=f"Go (n={go_mask.sum()})", color='blue', linewidth=2)

    if nogo_mask.sum() > 0:
        nogo_evoked = epochs[nogo_mask].average()
        ch_idx = nogo_evoked.ch_names.index(ch)
        ax.plot(times * 1000, nogo_evoked.data[ch_idx] * 1e6,
                label=f"NoGo (n={nogo_mask.sum()})", color='red', linewidth=2)

    ax.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax.axhline(0, color='k', linestyle='-', alpha=0.3)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude (uV)")
    ax.set_title(f"Go vs NoGo at {ch} — sj{SJ_NUM:02d} {label}")
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print(f"\nGo trials: {go_mask.sum()}, NoGo trials: {nogo_mask.sum()}")

## 16. Summary & Comparison Helper

Use this cell to compare the notebook output with the pipeline's saved epochs.

In [ ]:
# ── Compare with pipeline output (optional) ──
# Set this to a run's data directory to compare against the pipeline's saved epochs
COMPARE_DIR = ""  # e.g., "/Users/madhav/PSY197B/runs/2026-05-15_1200_aLL_sj_test_run/data"

if COMPARE_DIR:
    pipeline_epo_path = os.path.join(COMPARE_DIR, f"sj{SJ_NUM:02d}_{label}_EEG_Prepro1-epo.fif")
    if os.path.exists(pipeline_epo_path):
        pipeline_epochs = mne.read_epochs(pipeline_epo_path, preload=True, verbose=False)
        print(f"Pipeline epochs: {len(pipeline_epochs)}")
        print(f"Notebook epochs: {len(epochs)}")
        print(f"Count match: {len(pipeline_epochs) == len(epochs)}")

        if len(pipeline_epochs) == len(epochs):
            # Compare data arrays
            diff = np.abs(pipeline_epochs.get_data() - epochs.get_data())
            print(f"\nMax absolute difference: {diff.max():.2e}")
            print(f"Mean absolute difference: {diff.mean():.2e}")

            if diff.max() > 1e-10:
                print("\nWARNING: Differences detected!")
                # Find which channels/trials differ most
                trial_diff = diff.mean(axis=(1, 2))
                worst_trial = np.argmax(trial_diff)
                print(f"  Worst trial: {worst_trial} (mean diff={trial_diff[worst_trial]:.2e})")

                chan_diff = diff.mean(axis=(0, 2))
                worst_chan = np.argmax(chan_diff)
                print(f"  Worst channel: {epochs.ch_names[worst_chan]} (mean diff={chan_diff[worst_chan]:.2e})")
            else:
                print("Epochs match exactly!")
    else:
        print(f"Pipeline output not found: {pipeline_epo_path}")
else:
    print("Set COMPARE_DIR above to compare with pipeline output")

# Final summary
print(f"\n{'='*60}")
print(f"PIPELINE SUMMARY — sj{SJ_NUM:02d} {label}")
print(f"{'='*60}")
print(f"Raw EEG file: sj{SJ_NUM:02d}_{label}.vhdr")
print(f"Sampling rate: {SFREQ_TARGET} Hz")
print(f"Filter: {FILTER_LOW}–{FILTER_HIGH} Hz")
print(f"Reference: {REF_CHANNELS}")
print(f"Artifact removal: {'GEDAI' if USE_GEDAI else 'ICA' if APPLY_ICA else 'None'}")
print(f"Bad channels detected: {bad_chans if bad_chans else 'None'}")
print(f"Epochs: {len(epochs)} (tmin={TMIN}, tmax={TMAX})")
print(f"Trial data rows: {len(trial_data) if trial_data is not None else 'FAILED'}")
print(f"{'='*60}")